# Soft Due-Date Extension — CAR-T Supply Chain Optimisation

This notebook replicates the original Pyomo formulation (`Version0_Ishipment.ipynb`) and adds **patient-specific soft due dates**.

For each patient `p` a nonneg lateness variable `LATE[p]` and a penalty weight `PEN[p]` are introduced.
The due date `DUE[p]` is a mutable parameter defaulting to the global max turnaround time `ND`.
The model stays feasible even when due dates are tight — lateness is allowed but penalised.

**Requirements:** `Data200_profileA.dat` in the working directory, Gurobi (or any MILP solver)

## 1 · Imports

In [1]:
from pyomo.environ import *
from pyomo.common.timing import TicTocTimer
from time import process_time
import numpy as np
import matplotlib.pyplot as plt
import random

## 2 · Model Construction

`build_model()` constructs the Pyomo `AbstractModel`. Key additions vs. the baseline:

| Element | Description |
|---|---|
| `DUE[p]` | Mutable parameter — patient-specific due date (days), default = `ND` |
| `PEN[p]` | Mutable parameter — lateness penalty weight, default = 0 |
| `LATE[p]` | Continuous variable — days by which `TRT[p]` exceeds `DUE[p]` |
| `LATECON` | Constraint — `LATE[p] >= TRT[p] - DUE[p]` |
| Objective | Extended with `sum PEN[p] * LATE[p]` |

In [2]:
def build_model():
    model = AbstractModel()

    # Sets
    model.c  = Set()
    model.h  = Set()
    model.j  = Set()
    model.m  = Set()
    model.p  = Set()
    model.t  = RangeSet(130)
    model.tt = Set(initialize=model.t)

    # Parameters
    model.CIM        = Param(model.m)
    model.FCAP       = Param(model.m)
    model.TT1        = Param(model.j)
    model.TT3        = Param(model.j)
    model.U1         = Param(model.c, model.m, model.j)
    model.U3         = Param(model.m, model.h, model.j)
    model.INC        = Param(model.p, model.c, model.t, initialize=0)
    model.CVM        = Param(model.m,
                             default={'m1': 20920, 'm2': 156900, 'm3': 52300,
                                      'm4': 20920, 'm5': 156900, 'm6': 52300})
    model.FMAX       = Param()
    model.FMIN       = Param()
    model.TAD        = Param(within=NonNegativeReals)
    model.TLS        = Param(within=NonNegativeReals)
    model.TMFE       = Param(default=7)
    model.TQC        = Param(default=7)
    model.C_material = Param(default=10476)
    model.CQC        = Param(default=9312)
    model.ND         = Param(default=18)

    # NEW: soft due-date parameters
    def _due_init(model, p): return model.ND
    model.DUE = Param(model.p, initialize=_due_init, mutable=True, within=NonNegativeReals)

    def _pen_init(model, p): return 0.0
    model.PEN = Param(model.p, initialize=_pen_init, mutable=True, within=NonNegativeReals)

    # Decision variables
    model.E1   = Var(model.m,                                         within=Binary)
    model.X1   = Var(model.c, model.m,                                within=Binary)
    model.X2   = Var(model.m, model.h,                                within=Binary)
    model.Y1   = Var(model.p, model.c, model.m, model.j, model.t,    within=Binary)
    model.Y2   = Var(model.p, model.m, model.h, model.j, model.t,    within=Binary)
    model.INH  = Var(model.p, model.h, model.t,                       within=NonNegativeIntegers)
    model.CTM  = Var(model.p,                                          within=NonNegativeReals)
    model.FTD  = Var(model.p, model.m, model.h, model.j, model.t,    within=NonNegativeReals)
    model.TTC  = Var(model.p,                                          within=NonNegativeReals)
    model.LSA  = Var(model.p, model.c, model.m, model.j, model.t,    within=NonNegativeReals)
    model.LSR  = Var(model.p, model.c, model.m, model.j, model.t,    within=NonNegativeReals)
    model.MSO  = Var(model.p, model.m, model.h, model.j, model.t,    within=NonNegativeReals)
    model.OUTC = Var(model.p, model.c, model.t,                        within=NonNegativeReals)
    model.OUTM = Var(model.p, model.m, model.t,                        within=NonNegativeReals)
    model.INM  = Var(model.p, model.m, model.t,                        within=NonNegativeReals)
    model.DURV = Var(model.p, model.m, model.t,                        within=NonNegativeReals)
    model.RATIO= Var(model.m, model.t,                                 within=NonNegativeReals)
    model.CAP  = Var(model.m, model.t)
    model.TRT  = Var(model.p)
    model.ATRT = Var()
    model.STT  = Var(model.p)
    model.CTT  = Var(model.p)
    model.LATE = Var(model.p, within=NonNegativeReals)  # NEW

    # Objective
    def obj_rule(model):
        return (sum(model.CTM[p] for p in model.p)
                + sum(model.TTC[p] for p in model.p)
                + (model.C_material + model.CQC) * len(model.p)
                + sum(model.PEN[p] * model.LATE[p] for p in model.p))
    model.obj = Objective(rule=obj_rule)

    # Constraints
    def C1_rule(model, p):
        return model.CTM[p] == sum((model.E1[m] * (model.CIM[m] + model.CVM[m]))
                                   * len(model.t) / len(model.p) for m in model.m)
    model.C1 = Constraint(model.p, rule=C1_rule)

    def C2_rule(model, p):
        return model.TTC[p] == (
            sum(model.Y1[p, c, m, j, t] * model.U1[c, m, j]
                for c in model.c for m in model.m for j in model.j for t in model.t)
            + sum(model.Y2[p, m, h, j, t] * model.U3[m, h, j]
                  for m in model.m for h in model.h for j in model.j for t in model.t))
    model.C2 = Constraint(model.p, rule=C2_rule)

    def RATIOEQ_rule(model, m, t):
        return model.RATIO[m, t] == sum(model.DURV[p, m, t] / model.FCAP[m] for p in model.p)
    model.RATIOEQ = Constraint(model.m, model.t, rule=RATIOEQ_rule)

    def MSBnew_rule(model, p, m, t):
        return model.DURV[p, m, t] == (
            sum(model.INM[p, m, tt - 1] - model.OUTM[p, m, tt]
                for tt in model.tt if tt <= t and tt > 1)
            + model.OUTM[p, m, t])
    model.MSBnew = Constraint(model.p, model.m, model.t, rule=MSBnew_rule)

    def MSB1_rule(model, p, c, t, tt):
        if tt == t + model.TLS: return model.INC[p, c, t] == model.OUTC[p, c, tt]
        return Constraint.Skip
    model.MSB1 = Constraint(model.p, model.c, model.t, model.tt, rule=MSB1_rule)

    def MSB3_rule(model, p, c, m, j, t, tt):
        if tt == t + model.TT1[j]: return model.LSR[p, c, m, j, t] == model.LSA[p, c, m, j, tt]
        return Constraint.Skip
    model.MSB3 = Constraint(model.p, model.c, model.m, model.j, model.t, model.tt, rule=MSB3_rule)

    def MSB7_rule(model, p, c, t):
        return model.OUTC[p, c, t] == sum(model.LSR[p, c, m, j, t] for m in model.m for j in model.j)
    model.MSB7 = Constraint(model.p, model.c, model.t, rule=MSB7_rule)

    def MSB5_rule(model, p, m, t):
        return model.INM[p, m, t] == sum(model.LSA[p, c, m, j, t] for c in model.c for j in model.j)
    model.MSB5 = Constraint(model.p, model.m, model.t, rule=MSB5_rule)

    def MSB2_rule(model, p, m, t, tt):
        if tt == t + model.TMFE: return model.INM[p, m, t] == model.OUTM[p, m, tt]
        return Constraint.Skip
    model.MSB2 = Constraint(model.p, model.m, model.t, model.tt, rule=MSB2_rule)

    def MSB8_rule(model, p, m, t, tt):
        if tt == t + model.TQC:
            return model.OUTM[p, m, t] == sum(model.MSO[p, m, h, j, tt] for h in model.h for j in model.j)
        return Constraint.Skip
    model.MSB8 = Constraint(model.p, model.m, model.t, model.tt, rule=MSB8_rule)

    def MSB4_rule(model, p, m, h, j, t, tt):
        if tt == t + model.TT3[j]: return model.MSO[p, m, h, j, t] == model.FTD[p, m, h, j, tt]
        return Constraint.Skip
    model.MSB4 = Constraint(model.p, model.m, model.h, model.j, model.t, model.tt, rule=MSB4_rule)

    def MSB6_rule(model, p, h, t):
        return model.INH[p, h, t] == sum(model.FTD[p, m, h, j, t] for m in model.m for j in model.j)
    model.MSB6 = Constraint(model.p, model.h, model.t, rule=MSB6_rule)

    def CAP1_rule(model, m, t):
        return model.CAP[m, t] == model.FCAP[m] - sum(
            model.INM[p, m, tt] for p in model.p for tt in model.tt
            if tt < t and tt >= t - model.TMFE)
    model.CAP1 = Constraint(model.m, model.t, rule=CAP1_rule)

    def CAPCON1_rule(model, m, t):
        return (sum(model.INM[p, m, t] for p in model.p)
                - sum(model.OUTM[p, m, t] for p in model.p) <= model.CAP[m, t])
    model.CAPCON1 = Constraint(model.m, model.t, rule=CAPCON1_rule)

    def CON1_rule(model): return sum(model.E1[m] for m in model.m) <= 2
    model.CON1 = Constraint(rule=CON1_rule)

    def CON2_rule(model, c, m):      return model.X1[c, m] <= model.E1[m]
    model.CON2 = Constraint(model.c, model.m, rule=CON2_rule)

    def CON3_rule(model, m, h):      return model.X2[m, h] <= model.E1[m]
    model.CON3 = Constraint(model.m, model.h, rule=CON3_rule)

    def CON4_rule(model, p, c, m, j, t): return model.Y1[p, c, m, j, t] <= model.X1[c, m]
    model.CON4 = Constraint(model.p, model.c, model.m, model.j, model.t, rule=CON4_rule)

    def CON5_rule(model, p, m, h, j, t): return model.Y2[p, m, h, j, t] <= model.X2[m, h]
    model.CON5 = Constraint(model.p, model.m, model.h, model.j, model.t, rule=CON5_rule)

    def CON6_rule(model, p):
        return sum(model.Y1[p, c, m, j, t]
                   for c in model.c for m in model.m for j in model.j for t in model.t) == 1
    model.CON6 = Constraint(model.p, rule=CON6_rule)

    def CON7_rule(model, p):
        return sum(model.Y2[p, m, h, j, t]
                   for m in model.m for h in model.h for j in model.j for t in model.t) == 1
    model.CON7 = Constraint(model.p, rule=CON7_rule)

    def DEM_rule(model):
        return sum(model.INH[p, h, t] for p in model.p for h in model.h for t in model.t) <= len(model.p)
    model.DEM = Constraint(rule=DEM_rule)

    def CON8_rule(model, p, c, m, j, t):  return model.LSR[p, c, m, j, t] >= model.Y1[p, c, m, j, t] * model.FMIN
    model.CON8  = Constraint(model.p, model.c, model.m, model.j, model.t, rule=CON8_rule)

    def CON9_rule(model, p, c, m, j, t):  return model.LSR[p, c, m, j, t] <= model.Y1[p, c, m, j, t] * model.FMAX
    model.CON9  = Constraint(model.p, model.c, model.m, model.j, model.t, rule=CON9_rule)

    def CON10_rule(model, p, m, h, j, t): return model.MSO[p, m, h, j, t] >= model.Y2[p, m, h, j, t] * model.FMIN
    model.CON10 = Constraint(model.p, model.m, model.h, model.j, model.t, rule=CON10_rule)

    def CON11_rule(model, p, m, h, j, t): return model.MSO[p, m, h, j, t] <= model.Y2[p, m, h, j, t] * model.FMAX
    model.CON11 = Constraint(model.p, model.m, model.h, model.j, model.t, rule=CON11_rule)

    def CON12_rule(model, p):
        return (sum(model.Y2[p, m, 'h1', j, t] for m in model.m for j in model.j for t in model.t)
                == sum(model.INC[p, 'c1', t] for t in model.t))
    model.CON12 = Constraint(model.p, rule=CON12_rule)

    def CON13_rule(model, p):
        return (sum(model.Y2[p, m, 'h2', j, t] for m in model.m for j in model.j for t in model.t)
                == sum(model.INC[p, 'c2', t] for t in model.t))
    model.CON13 = Constraint(model.p, rule=CON13_rule)

    def CON14_rule(model, p):
        return (sum(model.Y2[p, m, 'h3', j, t] for m in model.m for j in model.j for t in model.t)
                == sum(model.INC[p, 'c3', t] for t in model.t))
    model.CON14 = Constraint(model.p, rule=CON14_rule)

    def CON15_rule(model, p):
        return (sum(model.Y2[p, m, 'h4', j, t] for m in model.m for j in model.j for t in model.t)
                == sum(model.INC[p, 'c4', t] for t in model.t))
    model.CON15 = Constraint(model.p, rule=CON15_rule)

    def START_rule(model, p):
        return model.STT[p] == sum(model.INC[p, c, t] * t for c in model.c for t in model.t)
    model.START = Constraint(model.p, rule=START_rule)

    def END_rule(model, p):
        return model.CTT[p] == sum(model.INH[p, h, t] * t for h in model.h for t in model.t)
    model.END = Constraint(model.p, rule=END_rule)

    def TSEQ_rule(model, p):  return model.STT[p] <= model.CTT[p]
    model.TSEQ  = Constraint(model.p, rule=TSEQ_rule)

    def TIME_rule(model, p):  return model.TRT[p] == model.CTT[p] - model.STT[p]
    model.TIME  = Constraint(model.p, rule=TIME_rule)

    def ATIME_rule(model):
        return model.ATRT == sum(model.TRT[p] for p in model.p) / len(model.p)
    model.ATIME = Constraint(rule=ATIME_rule)

    # NEW: lateness constraint
    def LATE_rule(model, p): return model.LATE[p] >= model.TRT[p] - model.DUE[p]
    model.LATECON = Constraint(model.p, rule=LATE_rule)

    return model

## 3 · Solve Scenario

In [3]:
def solve_scenario(custom_due_dates=None, penalty_weights=None,
                   solver_name='gurobi', display_summary=True):
    """
    Build, optionally update, and solve the model.

    Parameters
    ----------
    custom_due_dates : dict, optional  {patient_id: due_date_days}
    penalty_weights  : dict, optional  {patient_id: penalty_weight}
    solver_name      : str
    display_summary  : bool
    """
    model    = build_model()
    instance = model.create_instance('Data200_profileA.dat')

    if custom_due_dates:
        for p, val in custom_due_dates.items():
            if p in instance.p: instance.DUE[p] = val

    if penalty_weights:
        for p, w in penalty_weights.items():
            if p in instance.p: instance.PEN[p] = w

    results = SolverFactory(solver_name).solve(instance, tee=False)

    if display_summary:
        ok = (results.solver.status == SolverStatus.ok and
              results.solver.termination_condition == TerminationCondition.optimal)
        if not ok: print('Warning: solver did not report an optimal solution.')
        total_cost = value(
            sum(instance.CTM[p] for p in instance.p) +
            sum(instance.TTC[p] for p in instance.p) +
            (instance.C_material + instance.CQC) * len(instance.p) +
            sum(instance.PEN[p] * instance.LATE[p] for p in instance.p)
        )
        print(f'Total cost:          {total_cost:.2f}')
        print(f'Average return time: {value(instance.ATRT):.2f} days')
        print(f'Total lateness:      {value(sum(instance.LATE[p] for p in instance.p)):.2f} days')

    return instance

## 4 · Urgency Grouping Utilities

| Function | Purpose |
|---|---|
| `assign_urgency_groups` | Split patients into high / medium / low tiers; return due-date and penalty dicts |
| `summarize_metrics_by_group` | Avg TRT, avg/max lateness per tier after solving |
| `solve_urgency_scenario` | End-to-end convenience wrapper |

In [4]:
def assign_urgency_groups(instance, high_fraction=0.2, med_fraction=0.5, low_fraction=0.3,
                          due_dates=None, penalties=None, random_seed=42):
    """
    Divide patients into urgency groups and return due-date / penalty dicts.
    Defaults: due_dates={'high':16,'medium':18,'low':20},
              penalties={'high':10000,'medium':3000,'low':500}
    """
    if due_dates is None: due_dates = {'high': 16, 'medium': 18, 'low': 20}
    if penalties is None: penalties = {'high': 10000, 'medium': 3000, 'low': 500}

    patients = list(instance.p)
    n = len(patients)
    if n == 0: return {}, {}, {}

    total = high_fraction + med_fraction + low_fraction
    if abs(total - 1.0) > 1e-8:
        high_fraction /= total
        med_fraction  /= total

    num_high = int(round(n * high_fraction))
    num_med  = int(round(n * med_fraction))

    random.seed(random_seed)
    shuffled = patients[:]
    random.shuffle(shuffled)

    group_map, custom_due, penalty_wt = {}, {}, {}
    for idx, p in enumerate(shuffled):
        g = 'high' if idx < num_high else ('medium' if idx < num_high + num_med else 'low')
        group_map[str(p)] = g
        custom_due[str(p)] = due_dates[g]
        penalty_wt[str(p)] = penalties[g]

    return group_map, custom_due, penalty_wt

In [5]:
def summarize_metrics_by_group(instance, group_map, display=True):
    """Compute and optionally print per-group metrics after solving."""
    summary = {g: {'count': 0, 'sum_trt': 0.0, 'sum_late': 0.0, 'max_late': 0.0, 'late_count': 0}
               for g in ('high', 'medium', 'low')}

    for p in instance.p:
        pid = str(p)
        if pid not in group_map: continue
        g    = group_map[pid]
        trt  = value(instance.TRT[p])
        late = value(instance.LATE[p])
        summary[g]['count']    += 1
        summary[g]['sum_trt']  += trt
        summary[g]['sum_late'] += late
        summary[g]['max_late']  = max(summary[g]['max_late'], late)
        if late > 1e-6: summary[g]['late_count'] += 1

    for info in summary.values():
        n = info['count']
        info['avg_trt']  = info['sum_trt']  / n if n else 0.0
        info['avg_late'] = info['sum_late'] / n if n else 0.0

    if display:
        print('\nUrgency group metrics:')
        print(f"{'Group':<10}{'Count':>8}{'Avg TRT':>12}{'Avg Late':>12}{'Max Late':>12}{'Late #':>10}")
        for g in ('high', 'medium', 'low'):
            i = summary[g]
            print(f"{g:<10}{i['count']:>8d}{i['avg_trt']:>12.2f}{i['avg_late']:>12.2f}{i['max_late']:>12.2f}{i['late_count']:>10d}")

    return summary

In [6]:
def solve_urgency_scenario(high_fraction=0.2, med_fraction=0.5, low_fraction=0.3,
                           due_dates=None, penalties=None, random_seed=42,
                           solver_name='gurobi', display_summary=True):
    """End-to-end: assign urgency groups -> solve -> return (instance, group_map)."""
    base_instance = build_model().create_instance('Data200_profileA.dat')
    group_map, custom_due, penalty_wt = assign_urgency_groups(
        base_instance,
        high_fraction=high_fraction, med_fraction=med_fraction, low_fraction=low_fraction,
        due_dates=due_dates, penalties=penalties, random_seed=random_seed
    )
    instance = solve_scenario(
        custom_due_dates=custom_due, penalty_weights=penalty_wt,
        solver_name=solver_name, display_summary=display_summary
    )
    return instance, group_map

## 5 · Run: Urgency-Aware Scenario

Default split: **20 % high / 50 % medium / 30 % low**

In [ ]:
print('=== Solving urgency-aware scenario ===')
instance, group_map = solve_urgency_scenario(display_summary=True)
summary = summarize_metrics_by_group(instance, group_map)

=== Solving urgency-aware scenario ===
